# 03 — Hosted Whisper API versus Whisper Large V3 Turbo

Compare **OpenAI `whisper-1`** against the existing V3 Turbo pilot on the same five code-switched validation clips and canonical mixed-script references. Run automatic language detection and explicit Nepali: **10 API requests**, covering 101.952 seconds of submitted audio in total. No GPU or local model download is needed; Colab handles dataset decoding and scoring, and OpenAI handles ASR.

**Model distinction:** the [API reference](https://developers.openai.com/api/reference/resources/audio/subresources/transcriptions/methods/create) identifies `whisper-1` as Whisper **V2**. This is a comparison of two systems, with different checkpoints and decoding implementations. It does not isolate hosting and does not establish that newer GPT transcription models perform better. The user requested Whisper, so no GPT model is substituted. `whisper-1` is scheduled to shut down February 26, 2027; see the [deprecation notice](https://developers.openai.com/api/docs/deprecations).

The [Whisper model page](https://developers.openai.com/api/docs/models/whisper-1) lists $0.006 per minute when this notebook was prepared (October 22, 2026): nominally about **$0.0102** for one complete run, before billing rounding or retries. Check current billing/account access. Running section 3.2 uploads these clips and incurs API usage. No reference transcripts, prompts, CMI labels, or prior model predictions are sent to the API.

Start in a **fresh Colab CPU session**. Install once, restart the session, then resume at **1.2**. Set a Colab Secret called `OPENAI_API_KEY` and enable notebook access, or use the hidden password prompt in section 3.1. Never put your key in a code cell or share it in chat.

Successful responses are saved immediately in `whisper_api_pilot_checkpoint.json` in the Colab working directory. Rerunning within that runtime skips completed requests. A timeout may still have been processed by the provider: failed/uncertain requests are not automatically retried. The checkpoint is not persistent across a deleted runtime; download it with the final ZIP.


### 1.1 Install Dependencies, Then Restart

These releases match the completed pilot where applicable. No Torch or Transformers import is required. After this cell finishes, choose **Runtime → Restart session**, then resume at section 1.2 without rerunning installation. Dataset dependencies may conflict with unused preinstalled Colab apps; retain the pinned versions for this notebook.


In [ ]:
%pip install -q "openai==2.54.0" "datasets==4.4.1" "huggingface-hub==0.36.2" "numpy==2.1.3" "soundfile==0.13.1" "jiwer==4.0.0"


### 1.2 Import Libraries

All evaluation runs use the same normalization as the completed V3 Turbo pilot. This cell makes no API calls.


In [ ]:
import hashlib
import io
import json
import math
import os
import platform
import sys
import time
import unicodedata
import zipfile
from datetime import datetime, timezone
from getpass import getpass
from importlib.metadata import version
from pathlib import Path

import numpy as np
import soundfile as sf
import openai
import huggingface_hub
from datasets import load_dataset
from IPython.display import Audio, display
from jiwer import cer, wer
from openai import OpenAI

assert version('openai') == openai.__version__ == '2.54.0', 'Run 1.1, restart the session, then resume at 1.2.'
assert version('huggingface-hub') == huggingface_hub.__version__ == '0.36.2', 'Run 1.1, restart the session, then resume at 1.2.'
package_versions = {name: version(name) for name in ['openai', 'datasets', 'huggingface-hub', 'numpy', 'soundfile', 'jiwer']}
print('Versions:', package_versions)
print('CPU runtime is sufficient. No API requests have been made.')


### 1.3 Load the Existing V3 Turbo Results

The original manifest, settings, and predictions are embedded as recorded data. The existing pilot ZIP does not need to be uploaded. References stay unchanged.


In [ ]:
baseline = json.loads('{"created_at_utc": "2026-10-22T08:49:26.480010+00:00", "dataset": {"id": "saileshbro/nepali-cs-asr", "revision": "dd00ca41dd38774cd6097c372c16939d7f3746df", "config": "default", "split": "validation"}, "model": {"id": "openai/whisper-large-v3-turbo", "revision": "41f01f3fe87f28c78e2fbf8b568835947dd65ed9"}, "selection": {"speech_type": "cs", "seed": 42, "count": 5, "shuffle_buffer": 1000}, "manifest": [{"transcription": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_devanagari": "होइन धेरै जसो स्टुडेन्ट के हुन्छन् तपाईँको सटुडय एकदम हारड गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_romanized": "hoina dherai jaso student ke hunchan tapaiko study ekdam hard garchan matlab dina rata 1 garchan uniharu", "cmi": 18.75, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "JF_HlxEXcHQ", "start_ms": 90896, "end_ms": 99488, "duration_s": 8.592, "clip_id": "validation/JF_HlxEXcHQ/90896-99488", "decoded_duration_s": 8.592, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "5d7728fbbec30b6f4f42110ecd48a76d19dea181673460552219a1f9beefcaa8"}, {"transcription": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "transcription_devanagari": "यसलाई ईसिलय यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त सो हाफ लाइफ को meaning के हो आधा हुने हो नि त सो तिम्रो लाइक", "transcription_romanized": "yaslai easily yatikai pani nikalna milne thiyo 25% bhaneko ho ni ta so half life ko meaning ke ho aadha hune ho ni ta so timro like", "cmi": 26.92, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 633264, "end_ms": 642192, "duration_s": 8.928, "clip_id": "validation/zJYS-XNDP2Q/633264-642192", "decoded_duration_s": 8.928, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "a19babdc7287647007c9675646769c74d8dd186b780fb5ce9ae8a74f887c52ef"}, {"transcription": "यो हामीले निकाल्ने हो नि त mass भनेको basically N N हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "transcription_devanagari": "यो हामीले निकाल्ने हो नि त म्यास भनेको बेसिकली एन एन हाम्रो कति निक्लियो यति निक्लियो इन्टु molar म्यास molar म्यास हाम्रो कति हुन्छ भन्दाखेरि", "transcription_romanized": "yo hamile nikalne ho ni ta mass bhaneko basically N N hamro kati nikliyo yati nikliyo into molar mass molar mass hamro kati huncha bhandakheri", "cmi": 36.0, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 1512240, "end_ms": 1525520, "duration_s": 13.28, "clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "decoded_duration_s": 13.28, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "574ae80ba5da885329f71a80a96040083e9b54e656a9f3f7f9aa15b8980c94b4"}, {"transcription": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "transcription_devanagari": "फिजिकल इन्टरप्रिटेसन यदि गर्न सक्यो भने होइन त्यसबाट हामी क्वेश्चन फ्रम द प्रेडिक्टेबल लास्ट यीयर को क्वेश्चन बाट गर्न सक्छौँ मार्कस ल्याउन सजिलो भइदिन्छ रेपिटिशन रेट होइन", "transcription_romanized": "physical interpretation yadi garna sakyo bhane hoina tyasbata hami question from the predictable last year ko question bata garna sakchau marks lyauna sajilo bhaidincha repetition rate hoina", "cmi": 44.44, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1205232, "end_ms": 1216352, "duration_s": 11.12, "clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "decoded_duration_s": 11.12, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "aae11f0b20a48ac8d2b029fe82fbc8b5cda3bc675594c692368a1ba2d30de2d9"}, {"transcription": "Electric charge field potential capacitor DC circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "transcription_devanagari": "इलेक्ट्रिक चार्ज फिल्ड पोटेनशल क्यापासिटर डीसी सर्किट च्याप्टर हरू छन् होइन जसमध्ये मोसटलय तपाईँको कन्सेप्ट", "transcription_romanized": "Electric charge field potential capacitor DC circuit chapter haru chan hoina jasamadhye mostly tapaiko concept", "cmi": 33.33, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1169296, "end_ms": 1178352, "duration_s": 9.056, "clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "decoded_duration_s": 9.056, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "52e59651d4a7af08a78092d441cebc8414550ca2721ceb5476644d8d1d8162a1"}], "summary": {"auto": {"clips": 5, "reference_words": 111, "wer": 2.8018018018018016, "cer": 1.8628571428571428, "outputs_without_eos": 1}, "forced_ne": {"clips": 5, "reference_words": 111, "wer": 1.117117117117117, "cer": 1.5257142857142858, "outputs_without_eos": 2}}, "generation_configs": {"auto": {"max_length": 448, "max_new_tokens": 440, "min_length": 0, "min_new_tokens": null, "early_stopping": false, "max_time": null, "stop_strings": null, "do_sample": false, "num_beams": 1, "use_cache": true, "cache_implementation": null, "cache_config": null, "return_legacy_cache": null, "prefill_chunk_size": null, "temperature": 1.0, "top_k": 50, "top_p": 1.0, "min_p": null, "typical_p": 1.0, "epsilon_cutoff": 0.0, "eta_cutoff": 0.0, "repetition_penalty": 1.0, "encoder_repetition_penalty": 1.0, "length_penalty": 1.0, "no_repeat_ngram_size": 0, "bad_words_ids": null, "renormalize_logits": false, "forced_bos_token_id": null, "forced_eos_token_id": null, "remove_invalid_values": false, "exponential_decay_length_penalty": null, "suppress_tokens": [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50359, 50360, 50361, 50362, 50363], "begin_suppress_tokens": [220, 50257], "sequence_bias": null, "token_healing": false, "guidance_scale": null, "watermarking_config": null, "num_return_sequences": 1, "output_attentions": false, "output_hidden_states": false, "output_scores": false, "output_logits": null, "return_dict_in_generate": false, "pad_token_id": 50257, "bos_token_id": 50257, "eos_token_id": 50257, "encoder_no_repeat_ngram_size": 0, "decoder_start_token_id": 50258, "is_assistant": false, "num_assistant_tokens": 20, "num_assistant_tokens_schedule": "constant", "assistant_confidence_threshold": 0.4, "prompt_lookup_num_tokens": null, "max_matching_ngram_size": null, "assistant_early_exit": null, "assistant_lookbehind": 10, "target_lookbehind": 10, "disable_compile": false, "low_memory": null, "penalty_alpha": null, "dola_layers": null, "diversity_penalty": 0.0, "num_beam_groups": 1, "constraints": null, "force_words_ids": null, "_from_model_config": false, "transformers_version": "4.57.1", "alignment_heads": [[2, 4], [2, 11], [3, 3], [3, 6], [3, 11], [3, 14]], "forced_decoder_ids": null, "is_multilingual": true, "lang_to_id": {"<|af|>": 50327, "<|am|>": 50334, "<|ar|>": 50272, "<|as|>": 50350, "<|az|>": 50304, "<|ba|>": 50355, "<|be|>": 50330, "<|bg|>": 50292, "<|bn|>": 50302, "<|bo|>": 50347, "<|br|>": 50309, "<|bs|>": 50315, "<|ca|>": 50270, "<|cs|>": 50283, "<|cy|>": 50297, "<|da|>": 50285, "<|de|>": 50261, "<|el|>": 50281, "<|en|>": 50259, "<|es|>": 50262, "<|et|>": 50307, "<|eu|>": 50310, "<|fa|>": 50300, "<|fi|>": 50277, "<|fo|>": 50338, "<|fr|>": 50265, "<|gl|>": 50319, "<|gu|>": 50333, "<|haw|>": 50352, "<|ha|>": 50354, "<|he|>": 50279, "<|hi|>": 50276, "<|hr|>": 50291, "<|ht|>": 50339, "<|hu|>": 50286, "<|hy|>": 50312, "<|id|>": 50275, "<|is|>": 50311, "<|it|>": 50274, "<|ja|>": 50266, "<|jw|>": 50356, "<|ka|>": 50329, "<|kk|>": 50316, "<|km|>": 50323, "<|kn|>": 50306, "<|ko|>": 50264, "<|la|>": 50294, "<|lb|>": 50345, "<|ln|>": 50353, "<|lo|>": 50336, "<|lt|>": 50293, "<|lv|>": 50301, "<|mg|>": 50349, "<|mi|>": 50295, "<|mk|>": 50308, "<|ml|>": 50296, "<|mn|>": 50314, "<|mr|>": 50320, "<|ms|>": 50282, "<|mt|>": 50343, "<|my|>": 50346, "<|ne|>": 50313, "<|nl|>": 50271, "<|nn|>": 50342, "<|no|>": 50288, "<|oc|>": 50328, "<|pa|>": 50321, "<|pl|>": 50269, "<|ps|>": 50340, "<|pt|>": 50267, "<|ro|>": 50284, "<|ru|>": 50263, "<|sa|>": 50344, "<|sd|>": 50332, "<|si|>": 50322, "<|sk|>": 50298, "<|sl|>": 50305, "<|sn|>": 50324, "<|so|>": 50326, "<|sq|>": 50317, "<|sr|>": 50303, "<|su|>": 50357, "<|sv|>": 50273, "<|sw|>": 50318, "<|ta|>": 50287, "<|te|>": 50299, "<|tg|>": 50331, "<|th|>": 50289, "<|tk|>": 50341, "<|tl|>": 50348, "<|tr|>": 50268, "<|tt|>": 50351, "<|uk|>": 50280, "<|ur|>": 50290, "<|uz|>": 50337, "<|vi|>": 50278, "<|yi|>": 50335, "<|yo|>": 50325, "<|yue|>": 50358, "<|zh|>": 50260}, "max_initial_timestamp_index": 50, "no_timestamps_token_id": 50364, "prev_sot_token_id": 50362, "return_timestamps": false, "task_to_id": {"transcribe": 50360, "translate": 50359}, "language": null, "task": "transcribe"}, "forced_ne": {"max_length": 448, "max_new_tokens": 440, "min_length": 0, "min_new_tokens": null, "early_stopping": false, "max_time": null, "stop_strings": null, "do_sample": false, "num_beams": 1, "use_cache": true, "cache_implementation": null, "cache_config": null, "return_legacy_cache": null, "prefill_chunk_size": null, "temperature": 1.0, "top_k": 50, "top_p": 1.0, "min_p": null, "typical_p": 1.0, "epsilon_cutoff": 0.0, "eta_cutoff": 0.0, "repetition_penalty": 1.0, "encoder_repetition_penalty": 1.0, "length_penalty": 1.0, "no_repeat_ngram_size": 0, "bad_words_ids": null, "renormalize_logits": false, "forced_bos_token_id": null, "forced_eos_token_id": null, "remove_invalid_values": false, "exponential_decay_length_penalty": null, "suppress_tokens": [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50359, 50360, 50361, 50362, 50363], "begin_suppress_tokens": [220, 50257], "sequence_bias": null, "token_healing": false, "guidance_scale": null, "watermarking_config": null, "num_return_sequences": 1, "output_attentions": false, "output_hidden_states": false, "output_scores": false, "output_logits": null, "return_dict_in_generate": false, "pad_token_id": 50257, "bos_token_id": 50257, "eos_token_id": 50257, "encoder_no_repeat_ngram_size": 0, "decoder_start_token_id": 50258, "is_assistant": false, "num_assistant_tokens": 20, "num_assistant_tokens_schedule": "constant", "assistant_confidence_threshold": 0.4, "prompt_lookup_num_tokens": null, "max_matching_ngram_size": null, "assistant_early_exit": null, "assistant_lookbehind": 10, "target_lookbehind": 10, "disable_compile": false, "low_memory": null, "penalty_alpha": null, "dola_layers": null, "diversity_penalty": 0.0, "num_beam_groups": 1, "constraints": null, "force_words_ids": null, "_from_model_config": false, "transformers_version": "4.57.1", "alignment_heads": [[2, 4], [2, 11], [3, 3], [3, 6], [3, 11], [3, 14]], "forced_decoder_ids": null, "is_multilingual": true, "lang_to_id": {"<|af|>": 50327, "<|am|>": 50334, "<|ar|>": 50272, "<|as|>": 50350, "<|az|>": 50304, "<|ba|>": 50355, "<|be|>": 50330, "<|bg|>": 50292, "<|bn|>": 50302, "<|bo|>": 50347, "<|br|>": 50309, "<|bs|>": 50315, "<|ca|>": 50270, "<|cs|>": 50283, "<|cy|>": 50297, "<|da|>": 50285, "<|de|>": 50261, "<|el|>": 50281, "<|en|>": 50259, "<|es|>": 50262, "<|et|>": 50307, "<|eu|>": 50310, "<|fa|>": 50300, "<|fi|>": 50277, "<|fo|>": 50338, "<|fr|>": 50265, "<|gl|>": 50319, "<|gu|>": 50333, "<|haw|>": 50352, "<|ha|>": 50354, "<|he|>": 50279, "<|hi|>": 50276, "<|hr|>": 50291, "<|ht|>": 50339, "<|hu|>": 50286, "<|hy|>": 50312, "<|id|>": 50275, "<|is|>": 50311, "<|it|>": 50274, "<|ja|>": 50266, "<|jw|>": 50356, "<|ka|>": 50329, "<|kk|>": 50316, "<|km|>": 50323, "<|kn|>": 50306, "<|ko|>": 50264, "<|la|>": 50294, "<|lb|>": 50345, "<|ln|>": 50353, "<|lo|>": 50336, "<|lt|>": 50293, "<|lv|>": 50301, "<|mg|>": 50349, "<|mi|>": 50295, "<|mk|>": 50308, "<|ml|>": 50296, "<|mn|>": 50314, "<|mr|>": 50320, "<|ms|>": 50282, "<|mt|>": 50343, "<|my|>": 50346, "<|ne|>": 50313, "<|nl|>": 50271, "<|nn|>": 50342, "<|no|>": 50288, "<|oc|>": 50328, "<|pa|>": 50321, "<|pl|>": 50269, "<|ps|>": 50340, "<|pt|>": 50267, "<|ro|>": 50284, "<|ru|>": 50263, "<|sa|>": 50344, "<|sd|>": 50332, "<|si|>": 50322, "<|sk|>": 50298, "<|sl|>": 50305, "<|sn|>": 50324, "<|so|>": 50326, "<|sq|>": 50317, "<|sr|>": 50303, "<|su|>": 50357, "<|sv|>": 50273, "<|sw|>": 50318, "<|ta|>": 50287, "<|te|>": 50299, "<|tg|>": 50331, "<|th|>": 50289, "<|tk|>": 50341, "<|tl|>": 50348, "<|tr|>": 50268, "<|tt|>": 50351, "<|uk|>": 50280, "<|ur|>": 50290, "<|uz|>": 50337, "<|vi|>": 50278, "<|yi|>": 50335, "<|yo|>": 50325, "<|yue|>": 50358, "<|zh|>": 50260}, "max_initial_timestamp_index": 50, "no_timestamps_token_id": 50364, "prev_sot_token_id": 50362, "return_timestamps": false, "task_to_id": {"transcribe": 50360, "translate": 50359}, "language": "ne", "task": "transcribe"}}, "generation_overrides": {"return_timestamps": false, "temperature": 0.0, "return_dict_in_generate": true}, "runtime": {"python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]", "platform": "Linux-6.6.122+-x86_64-with-glibc2.39", "torch": "2.11.0+cu130", "cuda": "13.0", "gpu": "Tesla T4", "dtype": "torch.float16"}, "source_archive": "whisper_pilot_20261022T084926480010Z.zip", "source_sha256": "ff3d8ffd26f70fd269d60b5b5cadc92c81f28dc2939b085d951d62f93db33cb0", "predictions": [{"clip_id": "validation/JF_HlxEXcHQ/90896-99488", "mode": "auto", "hypothesis": " आई ना देरी जोस्ट एस्टुडेंट के उन्जन तो पाई को एस्टडी एकदम हार्ड गर्शन मतलब दीन रात एक गर्शन हुनियारू", "normalized_reference": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "normalized_hypothesis": "आई ना देरी जोस्ट एस्टुडेंट के उन्जन तो पाई को एस्टडी एकदम हार्ड गर्शन मतलब दीन रात एक गर्शन हुनियारू", "wer": 0.9411764705882353, "cer": 0.6052631578947368}, {"clip_id": "validation/zJYS-XNDP2Q/633264-642192", "mode": "auto", "hypothesis": " So, we got to get out of 25% of the half-life companies. So, you are like...", "normalized_reference": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25 भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "normalized_hypothesis": "so we got to get out of 25 of the half life companies so you are like", "wer": 0.8518518518518519, "cer": 0.7789473684210526}, {"clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "mode": "auto", "hypothesis": " so mass basically yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon y", "normalized_reference": "यो हामीले निकाल्ने हो नि त mass भनेको basically n n हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "normalized_hypothesis": "so mass basically yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon yon y", "wer": 8.88, "cer": 5.815789473684211}, {"clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "mode": "auto", "hypothesis": " physical interpretation of the governance of your money I know this one I\'m a question from the predictable last year co-present butter governance of so marks level of social evidence are reputation rate I know", "normalized_reference": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "normalized_hypothesis": "physical interpretation of the governance of your money i know this one i m a question from the predictable last year co present butter governance of so marks level of social evidence are reputation rate i know", "wer": 1.0, "cer": 0.6533333333333333}, {"clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "mode": "auto", "hypothesis": " एलेक्टिक चार्ज फिल्ट पोटेंशी एक आपसीटर डीची सर्कित चैप्टर हरुट सान एन अ जस्मद्य वोस्ट एली तपाइंगो निएक आपको निएक आपको निएक आपको", "normalized_reference": "electric charge field potential capacitor dc circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "normalized_hypothesis": "एलेक्टिक चार्ज फिल्ट पोटेंशी एक आपसीटर डीची सर्कित चैप्टर हरुट सान एन अ जस्मद्य वोस्ट एली तपाइंगो निएक आपको निएक आपको निएक आपको", "wer": 1.5333333333333334, "cer": 1.0777777777777777}, {"clip_id": "validation/JF_HlxEXcHQ/90896-99488", "mode": "forced_ne", "hypothesis": " आईना धेरी जोस्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्�", "normalized_reference": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "normalized_hypothesis": "आईना धेरी जोस्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्ट्�", "wer": 1.0, "cer": 3.736842105263158}, {"clip_id": "validation/zJYS-XNDP2Q/633264-642192", "mode": "forced_ne", "hypothesis": " रेला इजिली एत्तिकेपन निगालना मिलनी थे 25 पर्षेंट माने आन्दा आफ्लाइफ कम्विनिंगे आदा वनियो अन्दा तिम्रो लाइक", "normalized_reference": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25 भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "normalized_hypothesis": "रेला इजिली एत्तिकेपन निगालना मिलनी थे 25 पर्षेंट माने आन्दा आफ्लाइफ कम्विनिंगे आदा वनियो अन्दा तिम्रो लाइक", "wer": 0.9259259259259259, "cer": 0.6947368421052632}, {"clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "mode": "forced_ne", "hypothesis": " यो आम्रे निगालनी अन्ता, सो मास मानिको बेसिकली यन, यन आम्रे गथि निगलियो, एति निगलियो, इन्टु मोलर मास, मोलर मास आम्रे गथि उन्च, गन्दा यहरी, आ,", "normalized_reference": "यो हामीले निकाल्ने हो नि त mass भनेको basically n n हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "normalized_hypothesis": "यो आम्रे निगालनी अन्ता सो मास मानिको बेसिकली यन यन आम्रे गथि निगलियो एति निगलियो इन्टु मोलर मास मोलर मास आम्रे गथि उन्च गन्दा यहरी आ", "wer": 1.0, "cer": 0.6578947368421053}, {"clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "mode": "forced_ne", "hypothesis": " प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन", "normalized_reference": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "normalized_hypothesis": "प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन प्रिटेशन", "wer": 1.4814814814814814, "cer": 1.9866666666666666}, {"clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "mode": "forced_ne", "hypothesis": " एलेक्टिक चार्ज फिल्ट पोटेंसिल कैपसिटर डीसी सर्किट चैप्टर हरुट सान एन जस्मध्य बोस्ट एले तपाइन को न्सेप्ट", "normalized_reference": "electric charge field potential capacitor dc circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "normalized_hypothesis": "एलेक्टिक चार्ज फिल्ट पोटेंसिल कैपसिटर डीसी सर्किट चैप्टर हरुट सान एन जस्मध्य बोस्ट एले तपाइन को न्सेप्ट", "wer": 1.1333333333333333, "cer": 0.8666666666666667}]}')

DATASET_ID = baseline['dataset']['id']
DATASET_REVISION = baseline['dataset']['revision']
SPLIT = baseline['dataset']['split']
expected_clips = {row['clip_id']: row for row in baseline['manifest']}
assert SPLIT == 'validation' and len(expected_clips) == len(baseline['manifest']) == 5
print('V3 Turbo baseline:', baseline['source_archive'])
print('Reference audio:', sum(row['decoded_duration_s'] for row in baseline['manifest']), 'seconds')


### 2.1 Audio Validation and Scoring Helpers

Reuse the existing audio checks, waveform hashing, and normalization. English words retain their original script in primary scoring.


In [ ]:
columns = ['audio', 'transcription', 'transcription_devanagari', 'transcription_romanized', 'cmi', 'speech_type', 'speaker_id', 'video_id', 'start_ms', 'end_ms', 'duration_s']

def normalize_text(text):
    text = unicodedata.normalize('NFC', text).lower()
    text = text.translate(str.maketrans('०१२३४५६७८९', '0123456789'))
    text = ''.join(' ' if unicodedata.category(char).startswith('P') else char for char in text)
    return ' '.join(text.split())

assert normalize_text('Hello,  SCHOOL!') == 'hello school'
assert normalize_text('नेपालमा १२ school छन्।') == 'नेपालमा 12 school छन्'
assert normalize_text('cafe\u0301') == normalize_text('café')
assert normalize_text('school') != normalize_text('स्कुल')
assert normalize_text(' ' + chr(9) + chr(10)) == ''
assert wer(['a b', 'c d'], ['a', 'c d e']) == 0.5
assert wer('a b', '') == 1.0
assert wer('a', 'a b c') == 2.0
assert cer('ab', 'a') == 0.5
print('Normalization and scoring self-checks passed.')

def prepare_clip(row):
    video_id = row.get('video_id')
    start, end = row.get('start_ms'), row.get('end_ms')
    if (
        not isinstance(video_id, str) or not video_id.strip()
        or any(not isinstance(value, int) or isinstance(value, bool) for value in (start, end))
        or not 0 <= start < end
    ):
        raise ValueError('Missing or invalid video_id/start_ms/end_ms; inspect clip provenance.')
    key = f'{SPLIT}/{video_id}/{start}-{end}'
    reference, cmi = row.get('transcription'), row.get('cmi')
    if not isinstance(reference, str) or not normalize_text(reference):
        raise ValueError(f'Empty or invalid reference for {key}.')
    if isinstance(cmi, bool) or not isinstance(cmi, (int, float)) or not math.isfinite(cmi) or not 0 <= cmi <= 50:
        raise ValueError(f'Invalid bilingual CMI for {key}: {cmi}.')
    encoded = row.get('audio')
    if not isinstance(encoded, dict) or not isinstance(encoded.get('bytes'), bytes) or not encoded['bytes']:
        raise ValueError(f'Expected embedded WAV bytes for {key}; inspect the dataset format.')
    waveform, sample_rate = sf.read(io.BytesIO(encoded['bytes']), dtype='float32', always_2d=True)
    if sample_rate != 16000 or waveform.shape[0] == 0 or waveform.shape[1] == 0 or not np.isfinite(waveform).all():
        raise ValueError(f'Invalid or non-finite audio for {key}; expected nonempty 16 kHz audio.')
    channels = waveform.shape[1]
    audio = waveform.mean(axis=1)
    duration = len(audio) / sample_rate
    expected = row.get('duration_s')
    if (
        not 0 < duration <= 30
        or isinstance(expected, bool) or not isinstance(expected, (int, float)) or not math.isfinite(expected)
        or abs(duration - expected) > 0.02
        or abs(duration - (end - start) / 1000) > 0.02
    ):
        raise ValueError(f'Audio duration does not match metadata/provenance for {key}.')
    identity = {name: row[name] for name in columns if name != 'audio'}
    identity.update(
        clip_id=key, decoded_duration_s=duration, sample_rate=sample_rate, source_channels=channels,
        audio_sha256=hashlib.sha256(audio.tobytes()).hexdigest(),
    )
    return {'identity': identity, 'audio': audio}


def inspect_clip(clip):
    identity, audio = clip['identity'], clip['audio']
    print('CLIP:', identity['clip_id'])
    print('REFERENCE:', identity['transcription'])
    print('CMI:', identity['cmi'], '| SPEECH TYPE:', identity['speech_type'])
    print('Waveform shape:', audio.shape, '| dtype:', audio.dtype, '| type:', type(audio))
    print('Source channels:', identity['source_channels'], '| Sample rate:', identity['sample_rate'])
    print('Dataset duration:', identity['duration_s'], '| Decoded duration:', identity['decoded_duration_s'])
    print('Min:', float(audio.min()), '| Max:', float(audio.max()))
    print('Mean absolute amplitude:', float(np.abs(audio).mean()))
    display(Audio(audio, rate=identity['sample_rate']))


### 2.2 Recover the Same Five Clips

Read the pinned validation split, verify every original waveform hash and metadata field, and retain the original WAV bytes for API upload. No resampling, volume adjustment, or lossy conversion is introduced. This scan may take a few minutes.


In [ ]:
stream = load_dataset(DATASET_ID, name='default', split=SPLIT, revision=DATASET_REVISION, streaming=True)
stream = stream.decode(False).select_columns(columns)
found = {}
for row in stream:
    key = f'{SPLIT}/{row["video_id"]}/{row["start_ms"]}-{row["end_ms"]}'
    if key not in expected_clips:
        continue
    if key in found:
        raise ValueError(f'Ambiguous duplicate provenance: {key}.')
    clip = prepare_clip(row)
    expected = expected_clips[key]
    if clip['identity'] != expected:
        raise ValueError(f'Audio hash or metadata changed for {key}; do not compare different inputs.')
    clip['upload_bytes'] = row['audio']['bytes']
    found[key] = clip
missing = set(expected_clips) - set(found)
if missing:
    raise ValueError(f'Missing original clips: {sorted(missing)}.')
clips = [found[row['clip_id']] for row in baseline['manifest']]
print('Verified five exact audio hashes and all recorded clip metadata.')


### 2.3 Prepare Requests and Local Checkpointing

Use `/v1/audio/transcriptions`, `model='whisper-1'`, `response_format='verbose_json'`, and `temperature=0`. For automatic detection omit `language`; for Nepali supply `language='ne'`. No answer-key context is sent. OpenAI documents that Whisper temperature zero can trigger automatic temperature fallback; provider decoding is not identical to the previous greedy V3 Turbo configuration. API EOS and full decoder settings are unavailable.

Only failed/uncertain requests require manual investigation. Successful responses will not be resent if this cell and the run cell are repeated. The checkpoint stores results, request IDs, settings, and clip identity; no client object, key, headers, or exception message is serialized. Local errors also remain visible rather than being silently converted into model scores.


In [ ]:
API_MODEL = 'whisper-1'
API_BASE_URL = 'https://api.openai.com/v1'
REQUEST_SETTINGS = {'model': API_MODEL, 'response_format': 'verbose_json', 'temperature': 0.0}
LANGUAGE_MODES = [('auto', None), ('forced_ne', 'ne')]
CHECKPOINT_PATH = Path('whisper_api_pilot_checkpoint.json')
run_spec = {
    'dataset': baseline['dataset'], 'manifest': [clip['identity'] for clip in clips],
    'request_settings': REQUEST_SETTINGS, 'language_modes': LANGUAGE_MODES,
    'payload_sha256': {clip['identity']['clip_id']: hashlib.sha256(clip['upload_bytes']).hexdigest() for clip in clips},
    'endpoint': API_BASE_URL + '/audio/transcriptions', 'max_retries': 0,
}
# JSON canonicalization makes tuples and their checkpoint-loaded lists equivalent.
run_spec = json.loads(json.dumps(run_spec))
expected_jobs = {(mode, clip['identity']['clip_id']) for mode, _ in LANGUAGE_MODES for clip in clips}


def score_text(reference, hypothesis):
    if not isinstance(hypothesis, str):
        raise ValueError('API response text must be a string; retain the response before investigating.')
    ref, hyp = normalize_text(reference), normalize_text(hypothesis)
    if not ref:
        raise ValueError('Empty reference cannot be scored.')
    return {
        'normalized_reference': ref, 'normalized_hypothesis': hyp,
        'wer': wer(ref, hyp), 'cer': cer(ref.replace(' ', ''), hyp.replace(' ', '')),
    }


def save_checkpoint(state):
    temporary = CHECKPOINT_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(state, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(CHECKPOINT_PATH)


def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'run_spec': run_spec, 'predictions': []}
    state = json.loads(CHECKPOINT_PATH.read_text(encoding='utf-8'))
    if state['run_spec'] != run_spec:
        raise ValueError('Checkpoint inputs/settings differ. Preserve it separately; do not overwrite or reuse it.')
    jobs = [(row['mode'], row['clip_id']) for row in state['predictions']]
    if len(set(jobs)) != len(jobs) or not set(jobs) <= expected_jobs:
        raise ValueError('Checkpoint has duplicate or unknown jobs.')
    for row in state['predictions']:
        if row['status'] not in {'ok', 'api_error', 'pending'}:
            raise ValueError('Unknown checkpoint status.')
        if any(row[key] != value for key, value in expected_clips[row['clip_id']].items()):
            raise ValueError('Checkpoint reference/audio identity differs from this manifest.')
    return state


def run_api_pilot():
    state = load_checkpoint()
    completed = {(row['mode'], row['clip_id']): row for row in state['predictions']}
    for mode, language in LANGUAGE_MODES:
        for number, clip in enumerate(clips, 1):
            identity = clip['identity']
            job = (mode, identity['clip_id'])
            if job in completed:
                if completed[job]['status'] != 'ok':
                    print('STOP: a prior request failed or has uncertain completion. Preserve/export the checkpoint and investigate before retrying.')
                    return state
                print('Already saved:', mode, identity['clip_id'])
                continue
            record = {**identity, 'mode': mode, 'status': 'pending', 'requested_at_utc': datetime.now(timezone.utc).isoformat()}
            state['predictions'].append(record)
            save_checkpoint(state)
            options = dict(REQUEST_SETTINGS)
            if language is not None:
                options['language'] = language
            started = time.perf_counter()
            try:
                response = client.audio.transcriptions.create(
                    file=(f'clip_{number:02d}.wav', clip['upload_bytes'], 'audio/wav'), **options,
                )
            except openai.OpenAIError as error:
                record.update(status='api_error', error_type=type(error).__name__,
                              http_status=getattr(error, 'status_code', None),
                              request_id=getattr(error, 'request_id', None),
                              request_latency_s=time.perf_counter() - started)
                save_checkpoint(state)
                print('API request stopped:', record['error_type'], '| HTTP status:', record['http_status'])
                print('401: check key/access; 429: check quota/billing/rate limits. No automatic retry was made. Run the export cell to retain partial results.')
                return state
            record.update(status='ok', response=response.model_dump(mode='json'),
                          request_id=getattr(response, '_request_id', None),
                          request_latency_s=time.perf_counter() - started)
            # Save the paid response before scoring so local failures cannot trigger a resend.
            save_checkpoint(state)
            completed[job] = record
            print('Saved:', mode, identity['clip_id'])
            print('API:', record['response'].get('text', '')[:240])
    return state


checkpoint = load_checkpoint()
print('Previously saved requests:', len(checkpoint['predictions']), '/ 10')
print('This cell has made no API requests.')


### 3.1 Read Your API Key Privately

In Colab's left sidebar, open **Secrets** (key icon), add `OPENAI_API_KEY`, and enable notebook access. Alternatively, this cell provides a hidden password prompt. Press Enter after pasting into that prompt. Do not replace this code with a literal key. This cell constructs a client but does not send a request.


In [ ]:
api_key = os.environ.get('OPENAI_API_KEY', '').strip()
if not api_key:
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            api_key = userdata.get('OPENAI_API_KEY').strip()
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
if not api_key:
    api_key = getpass('OpenAI API key (hidden): ').strip()
if not api_key:
    raise ValueError('No key supplied. Add it to Colab Secrets or rerun this cell.')
client = OpenAI(api_key=api_key, base_url=API_BASE_URL, max_retries=0, timeout=120.0)
del api_key
print('API client ready. Key is not printed or included in result exports.')


### 3.2 Run the Ten API Transcriptions

Running this cell sends the audio to OpenAI and uses your API billing. It resumes the local checkpoint. The same 50.976 seconds of audio is submitted once per language mode. No new clips are added. The secret is never sent as transcript context.

If interrupted, rerun this cell: completed responses are skipped. If it reports a pending/failed request, preserve the checkpoint and investigate before any retry because completion/billing may be uncertain. Downloading the final ZIP preserves that checkpoint if the Colab runtime is deleted.


In [ ]:
checkpoint = run_api_pilot()
print('Saved responses:', sum(row['status'] == 'ok' for row in checkpoint['predictions']), '/ 10')


### 4.1 Compare API Results with V3 Turbo

Compute corpus WER/CER using the same references and normalization. Display the full transcript pairs for inspection. An empty API transcript counts as deletions. API failures are reported explicitly, and an incomplete condition receives no headline score; the notebook does not silently compare a smaller/easier subset.

The V3 Turbo FP16 rerun matched all ten original token sequences, so the embedded original pilot is the comparison baseline. This comparison does not isolate checkpoint, provider preprocessing, or decoding. Request latency includes network/service time and is not GPU inference speed. Five clips cannot establish a CMI trend, general superiority, or a definitive best model.


In [ ]:
checkpoint = load_checkpoint()
predictions = []
for row in checkpoint['predictions']:
    if row['status'] == 'ok':
        hypothesis = row['response'].get('text')
        predictions.append({**row, 'hypothesis': hypothesis, **score_text(row['transcription'], hypothesis)})
summary = {}
comparison = []
for mode, _ in LANGUAGE_MODES:
    rows = [row for row in predictions if row['mode'] == mode]
    full_coverage = len(rows) == 5 and {row['clip_id'] for row in rows} == set(expected_clips)
    entry = {'successful_clips': len(rows), 'expected_clips': 5, 'complete': full_coverage}
    if full_coverage:
        refs, hyps = [r['normalized_reference'] for r in rows], [r['normalized_hypothesis'] for r in rows]
        entry.update(wer=wer(refs, hyps), cer=cer([r.replace(' ', '') for r in refs], [h.replace(' ', '') for h in hyps]),
                     request_latency_s=sum(r['request_latency_s'] for r in rows))
    summary[mode] = entry
    old_score = baseline['summary'][mode]
    comparison.append({'system': 'V3 Turbo (Colab)', 'mode': mode, 'wer': old_score['wer'], 'cer': old_score['cer'], 'clips': 5})
    comparison.append({'system': 'whisper-1 (API; V2)', 'mode': mode, 'wer': entry.get('wer'), 'cer': entry.get('cer'), 'clips': len(rows)})
print('System                         | mode      | clips | WER      | CER')
for row in comparison:
    word_score = 'incomplete' if row['wer'] is None else f"{row['wer']:.2%}"
    char_score = 'incomplete' if row['cer'] is None else f"{row['cer']:.2%}"
    print(f"{row['system']:30} | {row['mode']:9} | {row['clips']:5} | {word_score:8} | {char_score}")
old_predictions = {(row['mode'], row['clip_id']): row for row in baseline['predictions']}
for row in predictions:
    old_row = old_predictions[row['mode'], row['clip_id']]
    assert old_row['normalized_reference'] == row['normalized_reference']
    print('\n', row['mode'], '|', row['clip_id'], '| CMI', row['cmi'])
    print('REFERENCE:', row['transcription'])
    print('V3 TURBO:', old_row['hypothesis'])
    print('WHISPER API:', row['hypothesis'])
    print('WER change (API minus Turbo):', f"{100 * (row['wer'] - old_row['wer']):+.2f} percentage points")
failures = [{key: row.get(key) for key in ['clip_id', 'mode', 'status', 'error_type', 'http_status']} for row in checkpoint['predictions'] if row['status'] != 'ok']
print('Failed/uncertain requests:', failures)


### 4.2 Download the Results ZIP

Run this even after an API error to retain partial responses and the checkpoint. You may skip section 4.1 when diagnosing an unexpected response shape; raw responses are still preserved here. The ZIP contains JSON only: references, original baseline, request settings, API response bodies/IDs, safe error metadata, scores, package versions, and the checkpoint. It contains no audio, API key, environment variables, client objects, or request headers. Download and send this ZIP back for independent verification. Do not send the executed notebook containing private debugging output.


In [ ]:
checkpoint = load_checkpoint()
# Score from the checkpoint so export works after a partial run or skipped summary.
export_predictions = []
for row in checkpoint['predictions']:
    if row['status'] == 'ok' and isinstance(row['response'].get('text'), str):
        export_predictions.append({**row, 'hypothesis': row['response']['text'], **score_text(row['transcription'], row['response']['text'])})
    else:
        export_predictions.append(dict(row))
export_summary = {}
for mode, _ in LANGUAGE_MODES:
    rows = [r for r in export_predictions if r['mode'] == mode and 'wer' in r]
    entry = {'successful_clips': len(rows), 'expected_clips': 5, 'complete': len(rows) == 5 and {r['clip_id'] for r in rows} == set(expected_clips)}
    if entry['complete']:
        refs, hyps = [r['normalized_reference'] for r in rows], [r['normalized_hypothesis'] for r in rows]
        entry.update(wer=wer(refs, hyps), cer=cer([r.replace(' ', '') for r in refs], [h.replace(' ', '') for h in hyps]))
    export_summary[mode] = entry
created_at = datetime.now(timezone.utc)
bundle = {
    'created_at_utc': created_at.isoformat(), 'purpose': 'five-clip hosted Whisper V2 versus V3 Turbo validation pilot',
    'api_model': API_MODEL, 'run_spec': run_spec, 'dataset': baseline['dataset'],
    'manifest': [clip['identity'] for clip in clips], 'original_pilot': baseline,
    'predictions': export_predictions, 'summary': export_summary,
    'limitations': ['Different checkpoint and provider decoding, not an isolated hosting comparison.',
                    'Temperature zero may invoke provider fallback; full decoding settings and API checkpoint revision unavailable.',
                    'Request latency includes network/service time; not comparable with GPU inference time.',
                    'Five validation clips only; no CMI association or broad model-ranking conclusion.'],
    'runtime': {'python': sys.version, 'platform': platform.platform(), 'packages': package_versions},
}
archive_path = Path('whisper_api_pilot_' + created_at.strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
with zipfile.ZipFile(archive_path, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('results.json', json.dumps(bundle, ensure_ascii=False, indent=2, allow_nan=False))
    archive.writestr('checkpoint.json', json.dumps(checkpoint, ensure_ascii=False, indent=2, allow_nan=False))
print('Saved:', archive_path.resolve())
from google.colab import files
files.download(str(archive_path))
